# Descargar las capas del SIG de Asturias (paso 1: datos *raw*)

**Qué hace:** descarga capas oficiales del servidor SIG del Principado (SITPA-IDEAS) y las guarda
**tal cual**, en GeoJSON y con la fecha de descarga en el nombre. No limpia nada: eso es el paso 2.

**Por qué así:** los servicios cambian o se caen. Guardando una copia fechada, la evaluación del
TFG se puede repetir con los mismos datos y en la memoria se cita "datos descargados el día X".

**Qué descarga** (editable en `SERVICES`):
- `Sanidad/Edificios_Sanitarios`: hospitales, centros de salud, consultorios…
- `Sanidad/Farmacias_Botiquines`: farmacias y botiquines.
- `Visor/Educacion`: centros de enseñanza.
- `UnidadesAdministrativas`: **localidades** (puntos con código INE y habitantes), **parroquias** (SADEI) y límites.
- `Visor/Salud`, solo la capa 10: **zonas básicas de salud** (polígonos).
- `VisorRPGUR/Visor_RPGUR` (planeamiento urbanístico, otra ruta del servidor): **clasificación del suelo**,
  suelo urbanizable, **núcleos rurales**, áreas de prevención de riesgos y planeamiento general vigente.

Las tres primeras ya se descargaron el 11/10/2026. Para no repetirlas, comenta sus líneas en `SERVICES`.

**Cómo funciona:** cada servicio ArcGIS REST tiene una lista de capas (`/layers?f=json`). Para cada
capa se piden primero los identificadores de todos sus elementos y luego los elementos por bloques,
porque el servidor devuelve como máximo unos 2.000 por consulta.

Además del GeoJSON, guarda un `manifest_<fecha>.json` con la URL, la fecha y el número de elementos
de cada capa: es la "ficha" de la descarga para la memoria.

In [3]:
%pip install -q geopandas requests


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import json
import os
import re
import time
from datetime import date

import geopandas as gpd
import requests

# Base URL of the Asturias GIS server (ArcGIS REST).
BASE_URL = "https://sig.asturias.es/servicios/rest/services"

# Services to download: service -> list of layer ids, or None for all its feature layers.
SERVICES = {
    "Sanidad/Edificios_Sanitarios": None,
    "Sanidad/Farmacias_Botiquines": None,
    "Visor/Educacion": None,
    "UnidadesAdministrativas": None,  # localities (INE code + population), parishes, boundaries
    "Visor/Salud": [10],              # basic health zones (polygons); other layers repeat Sanidad/*
    # Urban planning lives under another path of the same server, so it goes as a full URL.
    # 41 land classification, 39 developable land, 37 rural settlements,
    # 22 risk prevention areas, 57 general plan in force.
    "https://sig.asturias.es/server/rest/services/VisorRPGUR/Visor_RPGUR": [41, 39, 37, 22, 57],
}

# Output folder. In Colab with Drive mounted, e.g. "/content/drive/MyDrive/tfg/data/raw".
OUTPUT_DIR = "data/raw"

# Download date, used in file names so each download is a frozen snapshot.
TODAY = date.today().isoformat()

# Elements requested per call (below the server limit of ~2,000).
CHUNK_SIZE = 500

HEADERS = {"User-Agent": "TFG-UOC-gis-agent/0.1 (academic project)"}

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Funciones de descarga

- `get_json`: hace una petición y reintenta si el servidor falla (pasa a menudo con servicios públicos).
- `list_layers`: devuelve las capas de un servicio, sin los "grupos" (carpetas que no tienen datos).
- `download_layer`: descarga todos los elementos de una capa en coordenadas WGS84 (lat/lon, EPSG:4326).

In [5]:
def get_json(url, params=None, retries=3, wait_s=10):
    """GET a URL and return the parsed JSON, retrying on network or server errors."""
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, params=params, headers=HEADERS, timeout=60)
            response.raise_for_status()
            data = response.json()
            # ArcGIS reports some errors with HTTP 200 and an "error" key in the body.
            if isinstance(data, dict) and "error" in data:
                raise RuntimeError(data["error"])
            return data
        except Exception as error:
            print(f"   intento {attempt}/{retries} fallido: {error}")
            if attempt == retries:
                raise
            time.sleep(wait_s)


def service_url(service):
    """Return the REST URL of a service, given as a path under BASE_URL or as a full URL."""
    return service if service.startswith("https://") else f"{BASE_URL}/{service}"


def list_layers(service):
    """Return the feature layers (id, name, geometry type) of a MapServer service."""
    info = get_json(f"{service_url(service)}/MapServer/layers", {"f": "json"})
    return [
        {"id": layer["id"], "name": layer["name"], "geometry": layer.get("geometryType")}
        for layer in info["layers"]
        if layer.get("type") == "Feature Layer"  # skip group layers (no data of their own)
    ]


def download_layer(service, layer_id):
    """Download every feature of a layer as a GeoJSON dict (EPSG:4326)."""
    query_url = f"{service_url(service)}/MapServer/{layer_id}/query"
    # 1) Ask only for the ids: this call is not limited by the server's max record count.
    ids = get_json(query_url, {"where": "1=1", "returnIdsOnly": "true", "f": "json"})
    object_ids = sorted(ids.get("objectIds") or [])
    features = []
    # 2) Fetch the features in chunks of ids.
    for start in range(0, len(object_ids), CHUNK_SIZE):
        chunk = object_ids[start:start + CHUNK_SIZE]
        page = get_json(query_url, {
            "objectIds": ",".join(map(str, chunk)),
            "outFields": "*",
            "outSR": 4326,
            "f": "geojson",
        })
        features.extend(page.get("features", []))
    return {"type": "FeatureCollection", "features": features}, len(object_ids)


def slug(text):
    """Turn a layer name into a safe file name: 'Centros de Salud' -> 'centros_de_salud'."""
    text = text.lower()
    for accented, plain in zip("áéíóúüñ", "aeiouun"):
        text = text.replace(accented, plain)
    return re.sub(r"[^a-z0-9]+", "_", text).strip("_")

## 2. Ver qué capas tiene cada servicio

Antes de descargar, imprime las capas. Si algún servicio falla aquí, el servidor no es accesible
desde donde ejecutas el cuaderno (anótalo: es un riesgo de datos).

In [6]:
layers_by_service = {}
for service, wanted_ids in SERVICES.items():
    print(f"── {service}")
    try:
        layers = list_layers(service)
    except Exception as error:
        print(f"   ✗ no se pudo leer el servicio: {error}\n")
        continue
    if wanted_ids is not None:
        layers = [layer for layer in layers if layer["id"] in wanted_ids]
    layers_by_service[service] = layers
    for layer in layers:
        print(f"   [{layer['id']}] {layer['name']}  ({layer['geometry']})")
    print()

── Sanidad/Edificios_Sanitarios
   [0] Hospitales  (esriGeometryPoint)
   [1] Centros de Salud  (esriGeometryPoint)
   [2] Consultorios Locales  (esriGeometryPoint)
   [3] Consultorios Perifericos  (esriGeometryPoint)
   [4] Centros de Salud Mental  (esriGeometryPoint)
   [5] Centros de Orientación Familiar  (esriGeometryPoint)
   [6] Centros de Atención Especializada  (esriGeometryPoint)

── Sanidad/Farmacias_Botiquines
   [0] Botiquines  (esriGeometryPoint)
   [1] Farmacias  (esriGeometryPoint)

── Visor/Educacion
   [0] Centros Educativos  (esriGeometryPoint)
   [1] Centros de Educación Ambiental  (esriGeometryPoint)
   [2] Centros Universitarios  (esriGeometryPoint)

── UnidadesAdministrativas
   [0] Asturias  (esriGeometryPolygon)
   [1] Localidades  (esriGeometryPoint)
   [2] Concejos  (esriGeometryPolygon)
   [3] Parroquias  (esriGeometryPolygon)

── Visor/Salud
   [10] Zonificación  (esriGeometryPolygon)



## 3. Descargar y guardar

Cada capa se guarda como `<servicio>__<capa>_<fecha>.geojson`. Si el número de elementos descargados
no coincide con el número de identificadores, se avisa: la descarga estaría incompleta.

In [7]:
manifest = {"download_date": TODAY, "base_url": BASE_URL, "layers": []}

for service, layers in layers_by_service.items():
    for layer in layers:
        name = f"{slug(service.split('/')[-1])}__{slug(layer['name'])}"
        path = os.path.join(OUTPUT_DIR, f"{name}_{TODAY}.geojson")
        print(f"↓ {service} / {layer['name']}")
        try:
            collection, expected = download_layer(service, layer["id"])
        except Exception as error:
            print(f"   ✗ falló: {error}\n")
            manifest["layers"].append({"service": service, "layer_id": layer["id"],
                                       "layer_name": layer["name"], "error": str(error)})
            continue
        with open(path, "w", encoding="utf-8") as f:
            json.dump(collection, f, ensure_ascii=False)
        got = len(collection["features"])
        warning = "" if got == expected else f"  ⚠ esperados {expected}"
        print(f"   ✓ {got} elementos → {path}{warning}\n")
        manifest["layers"].append({
            "service": service, "layer_id": layer["id"], "layer_name": layer["name"],
            "url": f"{service_url(service)}/MapServer/{layer['id']}",
            "features": got, "expected": expected, "file": os.path.basename(path),
        })

manifest_path = os.path.join(OUTPUT_DIR, f"manifest_{TODAY}.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print("Ficha de la descarga:", manifest_path)

↓ Sanidad/Edificios_Sanitarios / Hospitales
   ✓ 14 elementos → data/raw/edificios_sanitarios__hospitales_2026-10-11.geojson

↓ Sanidad/Edificios_Sanitarios / Centros de Salud
   ✓ 69 elementos → data/raw/edificios_sanitarios__centros_de_salud_2026-10-11.geojson

↓ Sanidad/Edificios_Sanitarios / Consultorios Locales
   ✓ 34 elementos → data/raw/edificios_sanitarios__consultorios_locales_2026-10-11.geojson

↓ Sanidad/Edificios_Sanitarios / Consultorios Perifericos
   ✓ 111 elementos → data/raw/edificios_sanitarios__consultorios_perifericos_2026-10-11.geojson

↓ Sanidad/Edificios_Sanitarios / Centros de Salud Mental
   ✓ 24 elementos → data/raw/edificios_sanitarios__centros_de_salud_mental_2026-10-11.geojson

↓ Sanidad/Edificios_Sanitarios / Centros de Orientación Familiar
   ✓ 5 elementos → data/raw/edificios_sanitarios__centros_de_orientacion_familiar_2026-10-11.geojson

↓ Sanidad/Edificios_Sanitarios / Centros de Atención Especializada
   ✓ 7 elementos → data/raw/edificios_sanitarios_

## 4. Comprobación rápida

Abre cada fichero descargado con GeoPandas y muestra cuántos elementos tiene, sus columnas y las
dos primeras filas. Así ves qué campos trae cada capa antes de diseñar la limpieza (paso 2).

In [8]:
for entry in manifest["layers"]:
    if "file" not in entry:
        continue
    gdf = gpd.read_file(os.path.join(OUTPUT_DIR, entry["file"]))
    print(f"── {entry['layer_name']}: {len(gdf)} elementos | geometría {gdf.geom_type.unique().tolist()}")
    print("   columnas:", [c for c in gdf.columns if c != "geometry"])
    display(gdf.drop(columns="geometry").head(2))

── Hospitales: 14 elementos | geometría ['Point']
   columnas: ['objectid', 'id', 'nombre', 'area_sn']


,objectid,id,nombre,area_sn
0,1,113,Instituto Nacional de Silicosis,AREA II
1,2,110,Hospital Universitario Central de Asturias,AREA II


── Centros de Salud: 69 elementos | geometría ['Point']
   columnas: ['objectid', 'sipres', 'nombre', 'zona_3', 'area_sn']


,objectid,sipres,nombre,zona_3,area_sn
0,1,10175,C.S. TAPIA,B1.04,I
1,2,10177,C.S. VEGADEO,B1.05,I


── Consultorios Locales: 34 elementos | geometría ['Point']
   columnas: ['objectid', 'sipres', 'nombre', 'zona_3']


...

,objectid,codigo_ine,concejo,conceyu,st_area(shape),st_length(shape)
0,79,33001,ALLANDE,ALLANDE,3.398596e+08,122852.581705
1,89,33010,CANDAMO,CANDAMU,7.202583e+07,40633.685134


── Parroquias: 859 elementos | geometría ['Polygon', 'MultiPolygon']
   columnas: ['objectid', 'concejo', 'st_area(shape)', 'st_length(shape)', 'code', 'topo_ofi']


,objectid,concejo,st_area(shape),st_length(shape),code,topo_ofi
0,1,ALLANDE,2.897126e+07,28398.857470,010100,BERDUCEDO
1,2,ALLANDE,3.781990e+07,33542.927822,010500,LAGO


── Zonificación: 84 elementos | geometría ['Polygon', 'MultiPolygon']
   columnas: ['objectid', 'nombre', 'zona_1', 'zona_3', 'demarcacion', 'tipo', 'cabecera', 'area_sn', 'distrito', 'c_salud', 'c_local', 'c_perif', 'decreto', 'doc_c', 'hospi_zon', 'st_area(shape)', 'st_length(shape)']


,objectid,nombre,zona_1,zona_3,demarcacion,tipo,cabecera,area_sn,distrito,c_salud,c_local,c_perif,decreto,doc_c,hospi_zon,st_area(shape),st_length(shape)
0,1,Zona Básica de Salud II.27 Alto Aller-Cabanaqu...,ZBS.II.27,B2.27,Comprende el territorio del concejo de Aller f...,Z.B.S.,Oviedo. Hospital de referencia del Área: Hospi...,ÁREA II CENTRO-SUROCCIDENTE,,Cabanaquinta/Cabañaquinta,,"Col.lanzo, Piñeres, Casomera y Felechosa",https://miprincipado.asturias.es/bopa/2025/06/...,,HOSPITAL VITAL ÁLVAREZ BUYLLA,2.665910e+08,90479.503274
1,9,Zona Básica de Salud I.14 Castrillón,ZBS.I.14,B1.14,Comprende el territorio del concejo de Castril...,Z.B.S.,Avilés. Hospital de referencia del Área: Hospi...,ÁREA I OCCIDENTE,,Piedrasblancas.,,Raíces Nuevo/La Fundición-Salinas.,https://miprincipado.asturias.es/bopa/2025/06/...,,HOSPITAL UNIVERSITARIO SAN AGUSTÍN,5.266150e+07,49545.930125
